# Mobile App for Lottery Addiction

## Introduction

This report is to support the creation of a mobile application by a medical institute to help lottery addicts better estimate their chances of winning a lottery. The app will be a dedicated mobile app aiming to prevent and treat gambling addictions by answering questions such as:

- What is the probability of winning the big prize with a single ticket?
- What is the probability of winning the big prize with 40 (or any other amount) tickets?
- What is the probability of having at least five (or four, or three or two) winning numbers on a single ticket?

The report will also look at [data](https://www.kaggle.com/datasets/datascienceai/lottery-dataset) from the 6/49 lottery game in Canada for 3,665 draws, dating from 1982 to 2018.

## Core Functions

Probability and combinations are going to be calculated a number of times in this report, therefore, the first thing to do is to create functions that will help to carry out these calculations efficiently.

The first function is to calculate factorials, including building up a dictionary of results to avoid repeated calculations. The second function makes use of the first to calculate combinations.

In [1]:
factorials = {}
def factorial(n):
    if n <= 1:
        return 1
    if n not in factorials:
        factorials[n] = n * factorial(n - 1)
    return factorials[n]

In [2]:
def combinations(n, k):
    return int(factorial(n) / (factorial(k) * factorial(n - k)))

## Single Ticket Probability

The next function will calculate the probability that 6 unique numbers would win the main prize in the lottery.

In [3]:
def one_ticket_probability(numbers_list):
    # Verify that there are six unique numbers in the list
    numbers_count = len(numbers_list)
    numbers_set_count = len(set(numbers_list))
    if numbers_count == 6 and numbers_set_count == 6:
        outcomes = combinations(49, 6)
        pct_success = 100 * (1 / outcomes)
        str_numbers = ', '.join(str(x) for x in sorted(numbers_list))
        print("There is a 1 in {:,} chance ({:.2E}%) that a ticket with numbers {} will win the main prize in the next draw."
              .format(outcomes, pct_success, str_numbers))
    else:
        print("The list of numbers was not valid.")

Test the function with a selection of random sets of six numbers - all results should be the same:

In [4]:
from random import sample
test_numbers = []
for i in range(6):
    test_numbers.append(sample(range(1, 50), 6))
for test in test_numbers:
    one_ticket_probability(test)

There is a 1 in 13,983,816 chance (7.15E-06%) that a ticket with numbers 2, 12, 31, 35, 47, 49 will win the main prize in the next draw.
There is a 1 in 13,983,816 chance (7.15E-06%) that a ticket with numbers 2, 8, 23, 39, 41, 49 will win the main prize in the next draw.
There is a 1 in 13,983,816 chance (7.15E-06%) that a ticket with numbers 10, 19, 20, 23, 31, 37 will win the main prize in the next draw.
There is a 1 in 13,983,816 chance (7.15E-06%) that a ticket with numbers 13, 27, 31, 36, 46, 49 will win the main prize in the next draw.
There is a 1 in 13,983,816 chance (7.15E-06%) that a ticket with numbers 4, 9, 24, 31, 33, 39 will win the main prize in the next draw.
There is a 1 in 13,983,816 chance (7.15E-06%) that a ticket with numbers 11, 14, 22, 24, 28, 41 will win the main prize in the next draw.


## Historical Data Check for Canada Lottery

Use the historical data of the [Canad 6/49 Lottery](https://www.kaggle.com/datasets/datascienceai/lottery-dataset) to allow users to establish whether their numbers would have won a previous draw.

In [5]:
import pandas as pd

In [6]:
df = pd.read_csv('649.csv')

In [7]:
print("Number of rows: {}".format(df.shape[0]))
print("Number of columns: {}".format(df.shape[1]))

Number of rows: 3665
Number of columns: 11


In [8]:
df.head(3)

,PRODUCT,DRAW NUMBER,SEQUENCE NUMBER,DRAW DATE,NUMBER DRAWN 1,NUMBER DRAWN 2,NUMBER DRAWN 3,NUMBER DRAWN 4,NUMBER DRAWN 5,NUMBER DRAWN 6,BONUS NUMBER
0,649,1,0,6/12/1982,3,11,12,14,41,43,13
1,649,2,0,6/19/1982,8,33,36,37,39,41,9
2,649,3,0,6/26/1982,1,6,23,24,27,39,34


In [9]:
df.tail(3)

,PRODUCT,DRAW NUMBER,SEQUENCE NUMBER,DRAW DATE,NUMBER DRAWN 1,NUMBER DRAWN 2,NUMBER DRAWN 3,NUMBER DRAWN 4,NUMBER DRAWN 5,NUMBER DRAWN 6,BONUS NUMBER
3662,649,3589,0,6/13/2018,6,22,24,31,32,34,16
3663,649,3590,0,6/16/2018,2,15,21,31,38,49,8
3664,649,3591,0,6/20/2018,14,24,31,35,37,48,17


## Function for Historical Data Check

Create a function that will extract all of the six winning numbers from each draw in the dataset.

In [10]:
def extract_numbers(row):
    numbers = set()
    for i in range(1, 7):
        numbers.add(row['NUMBER DRAWN ' + str(i)])
    return numbers

In [11]:
previous_winning_numbers = df.apply(extract_numbers, axis=1)

Create a function to check a list of numbers from a ticket against the previous draws for wins.

In [12]:
def check_historical_occurence(ticket, draws):
    ticket = set(ticket)
    mask = draws == ticket
    times_won = mask.sum()
    if times_won == 0:
        print("Your numbers would not have won the main prize in previous draws.")
    elif times_won == 1:
        print("Your numbers won the main prize on {} occasion.".format(times_won))
    else:
        print("Your numbers won the main prize on {} occasions.".format(times_won))
    one_ticket_probability(ticket)
    print("Note: Previous resutls do not affect the chances of winning in any future draws.")

In [13]:
print(check_historical_occurence([6, 22, 24, 31, 32, 34], previous_winning_numbers))

Your numbers won the main prize on 1 occasion.
There is a 1 in 13,983,816 chance (7.15E-06%) that a ticket with numbers 6, 22, 24, 31, 32, 34 will win the main prize in the next draw.
Note: Previous resutls do not affect the chances of winning in any future draws.
None


## Multi-ticket Probability

Create a function to allow users to input the number of different tickets to play and it will calculate the odds of winning the main prize from those tickets. The number of tickets cannot exceed the maximum number of unique combinations of numbers (13,983,816) but must be more than 1 ticket.

In [14]:
def multi_ticket_probability(tickets):
    combos = combinations(49, 6)
    if tickets >= 1 and tickets <= combos:
        pct_win = 100 * (tickets / combos)
        plural = "s" if tickets > 1 else ""
        print("There is {}% chance that playing {} ticket{} will win the main prize in the next draw."
              .format(pct_win, tickets, plural))

In [15]:
test_tickets = [1, 10, 100, 10000, 1000000, 6991908, 13983816]
for test in test_tickets:
    multi_ticket_probability(test)

There is 7.151123842018516e-06% chance that playing 1 ticket will win the main prize in the next draw.
There is 7.151123842018517e-05% chance that playing 10 tickets will win the main prize in the next draw.
There is 0.0007151123842018516% chance that playing 100 tickets will win the main prize in the next draw.
There is 0.07151123842018516% chance that playing 10000 tickets will win the main prize in the next draw.
There is 7.151123842018517% chance that playing 1000000 tickets will win the main prize in the next draw.
There is 50.0% chance that playing 6991908 tickets will win the main prize in the next draw.
There is 100.0% chance that playing 13983816 tickets will win the main prize in the next draw.


## Fewer Winning Numbers - Function

Create a function that will calculate the chances of winning the smaller prizes by inputting the number of winning numbers per ticket. That is, 2, 3, 4 or 5 out of the 6 numbers match any of the 6 drawn numbers.

In [16]:
def probabilty_less_6(match_number):
    win_combos = combinations(6, match_number)
    other_combos = combinations(43, 6 - match_number)
    outcomes = win_combos * other_combos
    total_combos = combinations(49, 6)
    pct_win = 100 * (outcomes / total_combos)
    odds = round(total_combos / outcomes)
    print("""There is a {}% chance that this ticket has {} winning numbers.
That is, you have a 1 in {:,} chance of winning."""
         .format(pct_win, match_number, odds))

In [17]:
test_nums = [2, 3, 4, 5]
for test in test_nums:
    probabilty_less_6(test)
    print()

There is a 13.237802900152577% chance that this ticket has 2 winning numbers.
That is, you have a 1 in 8 chance of winning.

There is a 1.7650403866870101% chance that this ticket has 3 winning numbers.
That is, you have a 1 in 57 chance of winning.

There is a 0.0968619724401408% chance that this ticket has 4 winning numbers.
That is, you have a 1 in 1,032 chance of winning.

There is a 0.0018449899512407771% chance that this ticket has 5 winning numbers.
That is, you have a 1 in 54,201 chance of winning.

